<a href="https://colab.research.google.com/github/putri-indahsari/skripsi/blob/main/youtube_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Scrapping data komentar Makan Bergizi Gratis_Platform Youtube



In [2]:
pip install google-api-python-client pandas

In [ ]:
!pip install youtube-comment-downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 6.0 MB/s eta 0:00:00


In [ ]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
downloader = YoutubeCommentDownloader()
comments = []

video_url ="https://youtu.be/jsd410xprBA?si=KYUBOcvMbqourv6n"
max_comments = 2000

for comment in downloader.get_comments_from_url(video_url):
    comments.append([comment['text'], comment['time']])

## 1) Setup & Instalasi

In [ ]:
# Jalankan sel ini sekali untuk menginstal dependensi
!pip install google-api-python-client pandas python-dotenv tqdm tenacity

## 2) Import Library & Inisialisasi API

In [ ]:
import os
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple

import pandas as pd
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from dotenv import load_dotenv

# Muat API key dari .env (buat file .env di folder kerja dengan isi: YT_API_KEY=your_key_here)
load_dotenv()
API_KEY = os.getenv("YT_API_KEY")

if not API_KEY:
    raise ValueError("API key tidak ditemukan. Buat file .env dengan variabel YT_API_KEY atau set environment variable YT_API_KEY.")

# Membuat client YouTube API
YOUTUBE = build("youtube", "v3", developerKey=API_KEY)
print("YouTube API client siap.")


ValueError: API key tidak ditemukan. Buat file .env dengan variabel YT_API_KEY atau set environment variable YT_API_KEY.

## 3) Helper Functions (Paginasi & Retry)

In [ ]:

def _extract_items(resp: Dict) -> List[Dict]:
    return resp.get("items", []) if resp else []

@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=30),
    retry=retry_if_exception_type(HttpError),
)
def safe_execute(request):
    """Eksekusi request API dengan retry backoff untuk mengatasi error sementara / 429."""
    return request.execute()

def paginate(request_builder, **kwargs):
    """Generator untuk iterasi halaman API menggunakan nextPageToken."""
    next_page_token = None
    while True:
        request = request_builder(pageToken=next_page_token, **kwargs)
        resp = safe_execute(request)
        yield resp
        next_page_token = resp.get("nextPageToken")
        if not next_page_token:
            break


## 4) Fungsi: Pencarian Video (`search.list`)

In [ ]:

def search_videos(
    query: str,
    published_after: Optional[str] = None,
    published_before: Optional[str] = None,
    order: str = "date",
    region_code: Optional[str] = None,
    channel_id: Optional[str] = None,
    max_pages: int = 3,
    per_page: int = 50,
) -> pd.DataFrame:
    """Cari video berdasarkan kata kunci / channel dan kembalikan DataFrame hasil pencarian.

    Args:
        query: kata kunci pencarian (abaikan jika channel_id diberikan, bisa tetap diisi "")
        published_after: RFC3339, misal '2025-01-01T00:00:00Z'
        published_before: RFC3339
        order: 'date' | 'rating' | 'relevance' | 'title' | 'videoCount' | 'viewCount'
        region_code: kode negara (misal 'ID')
        channel_id: filter ke channel tertentu
        max_pages: batas jumlah halaman diproses (1 halaman = hingga 50 item)
        per_page: hasil per halaman (maks 50)
    """
    assert 0 < per_page <= 50, "per_page harus 1..50"
    collected = []
    pages = 0

    for resp in paginate(
        YOUTUBE.search().list,
        part="id,snippet",
        q=query if not channel_id else None,
        channelId=channel_id,
        type="video",
        maxResults=per_page,
        order=order,
        publishedAfter=published_after,
        publishedBefore=published_before,
        regionCode=region_code,
    ):
        pages += 1
        items = _extract_items(resp)
        for it in items:
            vid = it.get("id", {}).get("videoId")
            sn = it.get("snippet", {})
            collected.append({
                "video_id": vid,
                "published_at": sn.get("publishedAt"),
                "channel_id": sn.get("channelId"),
                "channel_title": sn.get("channelTitle"),
                "title": sn.get("title"),
                "description": sn.get("description"),
                "thumbnails_default": sn.get("thumbnails", {}).get("default", {}).get("url"),
                "thumbnails_high": sn.get("thumbnails", {}).get("high", {}).get("url"),
            })
        if pages >= max_pages:
            break

    df = pd.DataFrame(collected).drop_duplicates(subset=["video_id"]).reset_index(drop=True)
    return df


## 5) Fungsi: Detail Video (`videos.list`)

In [ ]:

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

def get_video_details(video_ids: List[str]) -> pd.DataFrame:
    """Ambil statistik & detail video (views, likes, tags, duration, dll). Batch 50 ID / request."""
    rows = []
    for chunk in tqdm(list(chunked(video_ids, 50)), desc="Ambil detail video"):
        resp = safe_execute(YOUTUBE.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(chunk)
        ))
        for it in _extract_items(resp):
            vid = it.get("id")
            sn = it.get("snippet", {})
            cd = it.get("contentDetails", {})
            st = it.get("statistics", {})
            rows.append({
                "video_id": vid,
                "category_id": sn.get("categoryId"),
                "tags": ",".join(sn.get("tags", [])) if sn.get("tags") else None,
                "duration_iso8601": cd.get("duration"),
                "dimension": cd.get("dimension"),
                "definition": cd.get("definition"),
                "caption": cd.get("caption"),
                "licensed_content": cd.get("licensedContent"),
                "projection": cd.get("projection"),
                "view_count": int(st.get("viewCount", 0)),
                "like_count": int(st.get("likeCount", 0)) if st.get("likeCount") is not None else None,
                "comment_count": int(st.get("commentCount", 0)) if st.get("commentCount") is not None else None,
            })
    return pd.DataFrame(rows)


## 6) Fungsi: Detail Channel (`channels.list`)

In [ ]:

def get_channel_details(channel_ids: List[str]) -> pd.DataFrame:
    """Ambil statistik channel (subscriberCount, videoCount, dll). Batch 50 ID / request."""
    rows = []
    for chunk in tqdm(list(chunked(channel_ids, 50)), desc="Ambil detail channel"):
        resp = safe_execute(YOUTUBE.channels().list(
            part="snippet,statistics,contentDetails",
            id=",".join(chunk)
        ))
        for it in _extract_items(resp):
            cid = it.get("id")
            sn = it.get("snippet", {})
            st = it.get("statistics", {})
            rows.append({
                "channel_id": cid,
                "channel_title": sn.get("title"),
                "channel_description": sn.get("description"),
                "country": sn.get("country"),
                "published_at_channel": sn.get("publishedAt"),
                "subscriber_count": int(st.get("subscriberCount", 0)) if st.get("hiddenSubscriberCount") is not True else None,
                "video_count": int(st.get("videoCount", 0)),
                "view_count_channel": int(st.get("viewCount", 0)),
            })
    return pd.DataFrame(rows)


## 7) Fungsi: Komentar (`commentThreads.list`)

In [ ]:

def get_comments(video_id: str, max_pages: int = 2, order: str = "relevance") -> pd.DataFrame:
    """Ambil komentar level-atas (top-level) untuk sebuah video.
    Gunakan max_pages untuk batasi kuota & waktu.
    order: 'time' atau 'relevance'
    """
    rows, pages = [], 0
    for resp in paginate(
        YOUTUBE.commentThreads().list,
        part="snippet",
        videoId=video_id,
        maxResults=100,
        order=order,
        textFormat="plainText",
    ):
        pages += 1
        for it in _extract_items(resp):
            sn = it.get("snippet", {})
            top = sn.get("topLevelComment", {}).get("snippet", {})
            rows.append({
                "video_id": video_id,
                "comment_id": it.get("id"),
                "comment_text": top.get("textDisplay"),
                "author": top.get("authorDisplayName"),
                "author_channel_id": top.get("authorChannelId", {}).get("value"),
                "like_count": top.get("likeCount"),
                "published_at": top.get("publishedAt"),
                "updated_at": top.get("updatedAt"),
                "total_reply_count": sn.get("totalReplyCount"),
            })
        if pages >= max_pages:
            break
    return pd.DataFrame(rows)


## 8) Contoh Alur Kerja Lengkap

In [ ]:

# Parameter contoh
QUERY = "teknologi AI Indonesia"
REGION = "ID"
DAYS_BACK = 30

published_after = (datetime.utcnow() - timedelta(days=DAYS_BACK)).strftime("%Y-%m-%dT%H:%M:%SZ")

# 1) Cari video terkait query (maks 2 halaman)
df_search = search_videos(
    query=QUERY,
    published_after=published_after,
    order="viewCount",
    region_code=REGION,
    max_pages=2,
    per_page=50,
)
print("Hasil pencarian:", df_search.shape)
df_search.head()


/tmp/ipykernel_27228/20447540.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  published_after = (datetime.utcnow() - timedelta(days=DAYS_BACK)).strftime("%Y-%m-%dT%H:%M:%SZ")


NameError: name 'YOUTUBE' is not defined

In [ ]:

# 2) Ambil detail video & gabungkan
if not df_search.empty:
    df_details = get_video_details(df_search['video_id'].dropna().tolist())
    df = df_search.merge(df_details, on="video_id", how="left")
else:
    df = df_search.copy()

print("Data gabungan:", df.shape)
df.head()


NameError: name 'df_search' is not defined

In [ ]:

# 3) Ambil detail channel untuk kanal yang muncul
if not df.empty:
    df_channels = get_channel_details(df['channel_id'].dropna().unique().tolist())
else:
    df_channels = pd.DataFrame()

print("Detail channel:", df_channels.shape)
df_channels.head()


In [ ]:

# 4) (Opsional) Ambil komentar dari video teratas berdasarkan views
COMMENTS_PAGES = 1  # batasi jumlah halaman komentar
if not df.empty:
    top_video = df.sort_values("view_count", ascending=False).iloc[0]["video_id"]
    df_comments = get_comments(top_video, max_pages=COMMENTS_PAGES, order="relevance")
else:
    df_comments = pd.DataFrame()

print("Komentar di 1 video:", df_comments.shape)
df_comments.head()


NameError: name 'df' is not defined

In [ ]:

# 5) Simpan ke file
out_dir = "output_youtube"
os.makedirs(out_dir, exist_ok=True)

csv_main = os.path.join(out_dir, "videos.csv")
csv_channels = os.path.join(out_dir, "channels.csv")
csv_comments = os.path.join(out_dir, "comments.csv")

df.to_csv(csv_main, index=False, encoding="utf-8")
df_channels.to_csv(csv_channels, index=False, encoding="utf-8")
df_comments.to_csv(csv_comments, index=False, encoding="utf-8")

print("Disimpan ke:", csv_main, csv_channels, csv_comments)


NameError: name 'df' is not defined


## 9) Tips Kuota & Etika Penggunaan
- **Kuota API**: Setiap endpoint memiliki biaya kuota. Misalnya `search.list` (100 unit), `videos.list` (1 unit), `commentThreads.list` (1 unit). Rancang jumlah halaman agar hemat kuota.
- **Batching**: Gabungkan hingga 50 `video_id`/`channel_id` per request untuk efisiensi.
- **Retry**: Notebook ini sudah menyertakan retry *exponential backoff* untuk error sementara.
- **Caching** (opsional): Simpan hasil *raw JSON* ke disk agar tidak memanggil API berulang.
- **Privasi & ToS**: Hanya kumpulkan data yang diperlukan, hormati privasi, dan patuhi kebijakan platform.



## 10) Variasi Penggunaan (Modifikasi Cepat)
- Filter ke **channel tertentu**: isi parameter `channel_id` pada `search_videos` dan kosongkan `query`.
- Ganti **periode**: set `published_after`/`published_before` dengan format RFC3339 (mis. `2025-08-01T00:00:00Z`).
- Ubah **pengurutan**: `order="viewCount"` untuk video paling banyak ditonton, atau `order="date"` untuk terbaru.
- Ambil **replies komentar**: gunakan endpoint `comments.list` dengan `parentId` dari komentar top-level (butuh fungsi tambahan).
- Gabungkan dengan **analisis teks**: setelah ekspor CSV, lakukan *cleaning* dan analisis sentimen sesuai kebutuhan riset Anda.



---

**Selesai.** Silakan jalankan sel dari atas ke bawah.  
Jika Anda butuh penyesuaian (misal: fokus channel tertentu, atau pipeline untuk monitoring berkala), tinggal beri tahu saya bagian mana yang ingin diubah.
